# oztab: a tour

This notebook walks the package from the bottom up: **the tableau is the primitive object**, and every number in the project is counted from it.

Run it with the repo's venv (`.venv/bin/python -m ipykernel ...`) or after `pip install -e .`.

The road map:

| § | What | Function |
|---|------|----------|
| 1 | Look at an actual tableau | `generate_tableaux` |
| 2 | Reproduce OZ Example 10 | `M_row` |
| 3 | The whole $h$-side table | `M_table` |
| 4 | The Schur side, via Jacobi–Trudi | `N_row` |
| 5 | The nonnegativity hypothesis | `negative_terms` |
| 6 | The profile decomposition | `profiles.*` |

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

from oztab import (generate_tableaux, is_valid, content_of,
                   M_row, M_table, format_row,
                   N_row, N_table, jacobi_trudi_terms, negative_terms, format_N_row)
from oztab.multisets import integer_partitions, multiset_partitions

## 1. The object

To expand $h_\lambda$ in the $\widetilde{s}$ basis, Orellana–Zabrocki count column-strict tableaux whose first row begins with $m$ **blank** cells ($m = |\lambda|$ for us), whose other cells hold **nonempty multisets**, with total content $\{\!\{1^{\lambda_1} 2^{\lambda_2}\cdots\}\!\}$. Such a $T$ contributes $\widetilde{s}_\mu$ where $\mu = \mathrm{shape}(T)$ **with its first row deleted**.

The one structural fact the whole package rests on: at $m = |\lambda|$ the body sits entirely *underneath* the blanks, and a blank is below every multiset in the order — so **no column constraint ever links the body to the first row**. A tableau is therefore just two independent pieces sharing a content:

- `first_row`: a multiset partition of some sub-content,
- `body`: a shape-$\mu$ multiset filling (rows weakly increasing, columns strictly increasing).

`generate_tableaux(content, mu)` enumerates exactly that split. Let's look at some.

In [2]:
content = content_of((2, 1))     # h_{2,1} -> content {{1,1,2}}
print("content of h_(2,1):", content)

tabs = generate_tableaux(content, (1,))   # body shape mu = (1)
print(f"\n{len(tabs)} tableaux with body shape (1):\n")
for T in tabs:
    print(T)          # compact view
    print()

content of h_(2,1): (1, 1, 2)

7 tableaux with body shape (1):

row1: {1} {1}
body: {2}

row1: {1,1}
body: {2}

row1: {1} {2}
body: {1}

row1: {1,2}
body: {1}

row1: {1}
body: {1,2}

row1: {2}
body: {1,1}

row1: (empty)
body: {1,1,2}



`render(faithful=True)` shows the $m$ blank cells (`.`), i.e. the actual OZ picture — the body really does hang under the blanks:

In [3]:
T = generate_tableaux(content_of((2, 1)), (2, 1))[0]
print(T.render(faithful=True))
print("\nshape_index (the mu indexing s~_mu):", T.shape_index())
print("content:", T.content())
print("passes independent validator:", is_valid(T))

. . .
{1} {1}
{2}

shape_index (the mu indexing s~_mu): (2, 1)
content: (1, 1, 2)
passes independent validator: True


## 2. Reproduce OZ Example 10

The paper's Example 10 expands $h_{(2,1)}$ and finds **20 tableaux** in total. `M_row` counts them, bucketed by $\mu$.

In [4]:
row = M_row((2, 1))
print(format_row((2, 1), row))
print("\ntotal tableaux:", sum(row.values()), "  (OZ Example 10 says 20)")

# the mu = () coefficient is exactly the number of multiset partitions of the content:
print("coeff of s~_():", row[()],
      "= #multiset partitions of", content_of((2, 1)),
      "=", len(multiset_partitions(content_of((2, 1)))))

h_21 = 4 s~_() + 7 s~_1 + 3 s~_11 + 4 s~_2 + 1 s~_21 + 1 s~_3

total tableaux: 20   (OZ Example 10 says 20)
coeff of s~_(): 4 = #multiset partitions of (1, 1, 2) = 4


## 3. The $h$-side table

`M_table(n)` gives every $h_\lambda$ with $|\lambda| = n$. All entries are **nonnegative** — they are counts of things.

In [5]:
for lam, row in M_table(4).items():
    print(format_row(lam, row))

h_4 = 5 s~_() + 7 s~_1 + 2 s~_11 + 5 s~_2 + 1 s~_21 + 2 s~_3 + 1 s~_4
h_31 = 7 s~_() + 14 s~_1 + 8 s~_11 + 10 s~_2 + 1 s~_111 + 4 s~_21 + 4 s~_3 + 1 s~_31 + 1 s~_4
h_22 = 9 s~_() + 17 s~_1 + 9 s~_11 + 14 s~_2 + 1 s~_111 + 6 s~_21 + 5 s~_3 + 1 s~_22 + 1 s~_31 + 1 s~_4
h_211 = 11 s~_() + 25 s~_1 + 18 s~_11 + 20 s~_2 + 4 s~_111 + 11 s~_21 + 7 s~_3 + 1 s~_211 + 1 s~_22 + 2 s~_31 + 1 s~_4
h_1111 = 15 s~_() + 37 s~_1 + 31 s~_11 + 31 s~_2 + 10 s~_111 + 20 s~_21 + 10 s~_3 + 1 s~_1111 + 3 s~_211 + 2 s~_22 + 3 s~_31 + 1 s~_4


## 4. The Schur side

The research object is $s_\lambda = \sum_\mu N_{\lambda,\mu}\, \widetilde{s}_\mu$. The tableau model only counts the $h$-side, so we get $N$ from $M$ by **Jacobi–Trudi**:

$$s_\lambda = \det\left(h_{\lambda_i - i + j}\right)_{1\le i,j\le \ell}$$

which expands into a *signed* sum of $h_\nu$'s. For two rows this is the familiar $s_{(a,b)} = h_a h_b - h_{a+1}h_{b-1}$ that the whole $b=2$/$b=3$ analysis was built on.

In [6]:
print("JT terms for s_(3,2):", jacobi_trudi_terms((3, 2)))
print("  i.e.  s_(3,2) = h_(3,2) - h_(4,1)\n")
print("JT terms for s_(2,1,1):", jacobi_trudi_terms((2, 1, 1)))

print()
for lam in integer_partitions(4):
    print(format_N_row(lam, N_row(lam)))

JT terms for s_(3,2): [(1, (3, 2)), (-1, (4, 1))]
  i.e.  s_(3,2) = h_(3,2) - h_(4,1)

JT terms for s_(2,1,1): [(1, (2, 1, 1)), (-1, (2, 2)), (-1, (3, 1)), (1, (4,))]

s_4 = 5 s~_() + 7 s~_1 + 2 s~_11 + 5 s~_2 + 1 s~_21 + 2 s~_3 + 1 s~_4
s_31 = 2 s~_() + 7 s~_1 + 6 s~_11 + 5 s~_2 + 1 s~_111 + 3 s~_21 + 2 s~_3 + 1 s~_31
s_22 = 2 s~_() + 3 s~_1 + 1 s~_11 + 4 s~_2 + 2 s~_21 + 1 s~_3 + 1 s~_22
s_211 = 1 s~_1 + 3 s~_11 + 1 s~_2 + 2 s~_111 + 2 s~_21 + 1 s~_211
s_1111 = 1 s~_111 + 1 s~_1111


## 5. The nonnegativity hypothesis

$N$ is a *signed* combination of nonnegative counts, so nothing forces $N_{\lambda,\mu} \ge 0$. That it seems to hold anyway is the hypothesis. `negative_terms` is the hunt; a single hit would be the headline result.

(For a full sweep use `scripts/sweep_nonneg.py --max-n 8`; cost grows ~7x per $n$.)

In [7]:
found = []
for n in range(8):
    for lam in integer_partitions(n):
        bad = negative_terms(N_row(lam))
        if bad:
            found.append((lam, bad))

print("negative coefficients for |lambda| <= 7:", found if found else "NONE")

negative coefficients for |lambda| <= 7: NONE


### Checking a proof: $N_{(n-1,1),\varnothing}$

`proofs/null_coefficient_S_n-1_1.tex` proves a closed form for the $\widetilde{s}_\varnothing$ coefficient of $s_{(n-1,1)}$:

$$N_{(n-1,1),\varnothing} \;=\; \sum_{k=0}^{n-1} p(k) \;-\; p(n)$$

via $s_{(n-1,1)} = h_{(n-1,1)} - h_{(n)}$ (Jacobi–Trudi), the fact that $[\widetilde{s}_\varnothing]\,h_\lambda$ is the number of **multiset partitions** of the content, and the count $\mathrm{mp}(\{\!\{1^{n-1},2\}\!\}) = \sum_{k=0}^{n-1} p(k)$. Nonnegativity comes from an explicit injection $\mathcal{P}(n) \hookrightarrow \mathcal{Q}(n)$ (turn one $1$ into a $2$ in a largest block).

This is the kind of thing the package is *for* — an independent check of every step:

In [8]:
from oztab.profiles import p

# Lemma 2 (Jacobi-Trudi):  s_(n-1,1) = h_(n-1,1) - h_(n)
print("JT terms for s_(6,1):", jacobi_trudi_terms((6, 1)), "  i.e. h_(6,1) - h_(7)\n")

# Proposition 3:  mp({{1^(n-1), 2}}) = sum_{k=0}^{n-1} p(k)
print("Prop 3:")
for n in range(2, 8):
    got = len(multiset_partitions(tuple([1] * (n - 1) + [2])))
    want = sum(p(k) for k in range(n))
    print(f"   n={n}: mp = {got:>3}   sum p(k) = {want:>3}   {got == want}")

# Theorem 2:  N_{(n-1,1),()} = sum_{k=0}^{n-1} p(k) - p(n)
print("\nTheorem 2:")
print(f"{'n':>4} {'oztab':>7} {'formula':>9} {'match':>7}")
for n in range(2, 12):
    lam = (n - 1, 1)
    got = N_row(lam).get((), 0)
    want = sum(p(k) for k in range(n)) - p(n)
    print(f"{n:>4} {got:>7} {want:>9} {str(got == want):>7}")

JT terms for s_(6,1): [(1, (6, 1)), (-1, (7,))]   i.e. h_(6,1) - h_(7)

Prop 3:
   n=2: mp =   2   sum p(k) =   2   True
   n=3: mp =   4   sum p(k) =   4   True
   n=4: mp =   7   sum p(k) =   7   True
   n=5: mp =  12   sum p(k) =  12   True
   n=6: mp =  19   sum p(k) =  19   True
   n=7: mp =  30   sum p(k) =  30   True

Theorem 2:
   n   oztab   formula   match
   2       0         0    True
   3       1         1    True
   4       2         2    True
   5       5         5    True
   6       8         8    True
   7      15        15    True
   8      23        23    True
   9      37        37    True
  10      55        55    True
  11      83        83    True


## 6. The profile decomposition

This is the older line of attack, aimed squarely at the **scalar** coefficient $N_{(a,b),\varnothing}$, now ported out of Sage into `oztab.profiles`.

For $\mu = \varnothing$ the tableau is a single row, and the count factors over the distinct parts $k$ of a partition $\lambda \vdash b$:

$$C(a,\lambda) = \sum_{J_0 + \sum_k J_k = a} p(J_0) \prod_k p_{\le n_k}(J_k), \qquad M_{(a,b),\varnothing} = \sum_{\lambda \vdash b} C(a,\lambda)$$

**Profile invariance:** $C(a,\lambda)$ depends only on the *multiset of multiplicities* $\{n_k\}$, never on the part values $k$ — read straight off the formula, since $k$ appears nowhere in it. So partitions of $b$ sharing a profile contribute equally, and the sum collapses onto distinct profiles.

In [9]:
from oztab.profiles import (profile, contribution, profile_counts,
                            M_empty_via_profiles, N_empty_via_profiles,
                            cross_check, assignment_search, p)

# profile invariance, concretely
print("profile((2,1)) =", profile((2, 1)), " profile((3,1)) =", profile((3, 1)))
print("C(5,(2,1)) =", contribution(5, (2, 1)), " C(5,(3,1)) =", contribution(5, (3, 1)))
print("profile((1,1)) =", profile((1, 1)), " profile((2,2)) =", profile((2, 2)))
print("C(5,(1,1)) =", contribution(5, (1, 1)), " C(5,(2,2)) =", contribution(5, (2, 2)))

# b = 5 is where the collapse first bites
print("\nb=5 profiles:", profile_counts(5), "  <- (1,1) and (2,1) each shared by 2 partitions")

profile((2,1)) = (1, 1)  profile((3,1)) = (1, 1)
C(5,(2,1)) = 45  C(5,(3,1)) = 45
profile((1,1)) = (2,)  profile((2,2)) = (2,)
C(5,(1,1)) = 28  C(5,(2,2)) = 28

b=5 profiles: {(1,): 1, (1, 1): 2, (2, 1): 2, (3, 1): 1, (5,): 1}   <- (1,1) and (2,1) each shared by 2 partitions


The profile formula and the tableau enumerator are **completely independent derivations** of the same number. That they agree is the strongest check in the repo:

In [10]:
print("profile formula vs tableau enumerator (a+b <= 8):",
      cross_check(8) or "ALL MATCH")

# the solved case b = 2:  N_{(a,2),()} = C(a,(1,1)) - p(a+1)
print(f"\n{'a':>3} {'N(a,2)':>8} {'C(a,(1,1)) - p(a+1)':>22}")
for a in range(2, 10):
    print(f"{a:>3} {N_row((a, 2)).get((), 0):>8} "
          f"{contribution(a, (1, 1)) - p(a + 1):>22}")

profile formula vs tableau enumerator (a+b <= 8): ALL MATCH

  a   N(a,2)    C(a,(1,1)) - p(a+1)
  2        2                      2
  3        4                      4
  4       10                     10
  5       17                     17
  6       32                     32
  7       51                     51


  8       84                     84


  9      128                    128


### Where $b = 3$ stands

The naive conjecture — that the Jacobi–Trudi leftover concentrates on the $(1^b)$ profile — **fails at $a = 10$**, where $N(10,3) = 347 > C(10,[1,1,1]) = 345$. So any correct formula must draw on the $(2,1)$ profile too. `assignment_search` ranks the profile-level injection schemes by $a_0$, the smallest $a$ from which the domination inequalities hold for good.

In [11]:
print(f"{'a':>4} {'C(a,[1,1,1])':>14} {'N(a,3)':>8} {'difference':>12}")
for a in range(3, 13):
    c = contribution(a, (1, 1, 1))
    n = N_empty_via_profiles(a, 3)
    print(f"{a:>4} {c:>14} {n:>8} {c - n:>12}")

print("\nbest injection schemes for b = 3 (lower a0 = works further down):")
for a0, scheme, leftover in assignment_search(3)[:3]:
    print(f"  a0 = {a0}: " + "; ".join(f"{src} -> {tgts}" for src, tgts in scheme))

   a   C(a,[1,1,1])   N(a,3)   difference
   3             10        2            8
   4             19       10            9
   5             33       20           13
   6             57       44           13
   7             92       76           16
   8            147      134           13
   9            227      216           11
  10            345      347           -2
  11            512      528          -16
  12            752      803          -51

best injection schemes for b = 3 (lower a0 = works further down):
  a0 = 2: (2,) -> [(2, 1)]; (1, 1) -> [(3,), (1, 1, 1)]
  a0 = 4: (2,) -> [(1, 1, 1)]; (1, 1) -> [(3,), (2, 1)]
  a0 = 6: (2,) -> [(3,), (1, 1, 1)]; (1, 1) -> [(2, 1)]


---

## Cheat sheet

```python
from oztab import (
    generate_tableaux,   # the objects: (content, mu) -> [MultisetTableau]
    count_tableaux,      # ... just the count, without building them (much faster)
    M_row, M_table,      # h-side: coefficients of s~_mu in h_lambda  (>= 0, counts)
    N_row, N_table,      # Schur side, via Jacobi-Trudi   (signed!)
    negative_terms,      # the nonnegativity hunt
    jacobi_trudi_terms,  # det expansion: lambda -> [(sign, nu)]
)
from oztab.profiles import contribution, profile, assignment_search, cross_check
```

```bash
pytest                                          # 14 tests, no Sage needed
python scripts/sweep_nonneg.py --max-n 8        # -> data/N_coefficients.csv
python scripts/profile_report.py --b 3 --search # the b=3 tables
sage -python scripts/sage_check.py 6            # external oracle (needs Sage)
```

**Cost:** the enumeration is exponential — roughly **7× per $n$**. $n\le 8$ is seconds, $n=9$ is minutes, $n=10$ is the better part of an hour. If you need to go further, that's an algorithms problem, not a patience problem.